# v9c CrossJEPA — Colab training notebook (Method 1)

Trains **Method 1** (3D volume → 2D slice with frozen v8 ConvNeXt-Tiny teacher).

**Inputs you need uploaded to Drive (once):**
1. `MyDrive/colab_bundle.zip` — source code (built by `scripts/rebundle_colab.py`)
2. `MyDrive/crossjepa_data.zip` — all 3D MRI data (built by `scripts/build_crossjepa_dataset_bundle.py`, ~13 GB, contains 800 healthy + 1251 BraTS-2021)

**Compute requirements**: Colab Pro+ with A100 (40 GB) or H100. T4 is too small for the full 3D ViT at `(144, 192, 192)` — drop to `(96, 128, 128)` if you must.

Pipeline:
1. Mount Drive + verify GPU
2. Install deps (`nibabel`, `segmentation-models-pytorch`, `timm`)
3. Unzip the source bundle from Drive
4. Unzip the dataset bundle from Drive
5. Pull the frozen v8 UNet checkpoint from HF Models (only network download)
6. Smoke-test the build on this GPU
7. Train Method 1 end-to-end

Method 2 (modality → modality) requires 4 per-modality I-JEPA teachers pretrained first; see [proposals/v9c_crossjepa_IMPLEMENTATION_STATUS.md](https://github.com/archisman-das/Neuro-Lens-AI/blob/main/proposals/v9c_crossjepa_IMPLEMENTATION_STATUS.md) for the runbook.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -10

## 2. Install dependencies

In [ ]:
%pip install -q nibabel==5.* huggingface_hub>=0.20 segmentation-models-pytorch>=0.3.3 timm>=0.9.16

## 3. Unzip the source bundle (from Drive)

Upload `colab_bundle.zip` (built locally by `scripts/rebundle_colab.py`) to `MyDrive/` once. This cell unzips it to `/content/neurolens/`.

In [ ]:
import os, sys
BUNDLE = '/content/drive/MyDrive/colab_bundle.zip'
DEST = '/content/neurolens'
!rm -rf {DEST}
!mkdir -p {DEST}
!unzip -q -o {BUNDLE} -d {DEST}
sys.path.insert(0, DEST)
os.chdir(DEST)
!ls src/research/v9c_crossjepa/
!ls src/train_v9c_*

## 4. Unzip the dataset bundle (from Drive)

Upload `crossjepa_data.zip` (built locally by `scripts/build_crossjepa_dataset_bundle.py`, ~13 GB) to `MyDrive/` once. This cell unzips both `healthy/` (800 radiata-ai T1) and `brats/` (1251 BraTS-2021 4-modality) to `/content/data/`.

In [ ]:
DATA_BUNDLE = '/content/drive/MyDrive/crossjepa_data.zip'
DATA_DEST = '/content/data'
!rm -rf {DATA_DEST}
!mkdir -p {DATA_DEST}
!unzip -q -o {DATA_BUNDLE} -d {DATA_DEST}
import subprocess
n_healthy = subprocess.run(['bash', '-c', f'find {DATA_DEST}/healthy -name "*.nii.gz" | wc -l'], capture_output=True, text=True).stdout.strip()
n_brats = subprocess.run(['bash', '-c', f'find {DATA_DEST}/brats -name "*.nii.gz" | wc -l'], capture_output=True, text=True).stdout.strip()
print(f'unzipped: {n_healthy} healthy + {n_brats} BraTS NIfTI files')

## 5. Pull the frozen v8 UNet checkpoint (one-time, ~383 MB)

This is the **only** network download in this notebook. The v8 PyTorch state dict lives in our public HF Models repo.

In [ ]:
from huggingface_hub import hf_hub_download
V8_CKPT = hf_hub_download(
    repo_id='Tubai01/neurolens-models', repo_type='model',
    filename='attention_unet_v8/best_micro.pt',
)
print(f'v8 ckpt at: {V8_CKPT}')
!ls -lh {V8_CKPT}

## 6. Smoke-test the build

Confirms the 3D ViT + predictor + frozen v8 teacher construct, a single forward+backward step works on this GPU, and the frozen-teacher invariant holds.

In [ ]:
import glob, torch, time
from torch.utils.data import DataLoader
from src.research.v9c_crossjepa.dataset_3d import Vol2SliceDataset
from src.research.v9c_crossjepa.v8_teacher import V8FrozenTeacher
from src.research.v9c_crossjepa.volume_to_slice import Vol2SliceModel

device = 'cuda'
torch.manual_seed(0)

# Load v8 frozen teacher
print('[smoke] loading v8 teacher...')
teacher = V8FrozenTeacher.from_unet_checkpoint(
    V8_CKPT,
    encoder_name='tu-convnext_tiny.fb_in22k_ft_in1k',
    in_channels=3, image_size=384, device=device,
)
snapshot = [p.clone() for p in teacher.encoder.parameters()]

# Build model
model = Vol2SliceModel(
    v8_teacher=teacher,
    volume_size=(144, 192, 192), in_chans=1, patch_size=16,
    encoder_dim=384, encoder_depth=12, encoder_heads=6,
    predictor_dim=192, predictor_depth=6,
).to(device)
print(f'[smoke] trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f} M')

# Single-step forward+backward on a tiny dataset slice
scans = sorted(glob.glob('/content/data/healthy/*/sub-*/ses-*/anat/*.nii.gz'))[:4]
print(f'[smoke] using {len(scans)} healthy scans for the smoke step')
ds = Vol2SliceDataset(scan_paths=scans, volume_size=(144, 192, 192),
                       in_channels=1, slices_per_volume=1)
def _coll(b):
    keys = ('volume','target_slice_rgb','plane_idx','slice_idx_norm','voxel_spacing','intensity_hist')
    return {k: torch.stack([s[k] for s in b]) for k in keys}
loader = DataLoader(ds, batch_size=1, collate_fn=_coll)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
for batch in loader:
    for k, v in batch.items():
        if torch.is_tensor(v): batch[k] = v.to(device, non_blocking=True)
    t0 = time.perf_counter()
    with torch.amp.autocast('cuda'):
        out = model.training_step(batch)
    out['loss'].backward(); opt.step(); opt.zero_grad()
    print(f'[smoke] step OK in {time.perf_counter()-t0:.2f}s  loss={float(out["loss"]):.4f}  cos_sim={float(out["cos_sim"]):.4f}')
    print(f'        gpu_mem = {torch.cuda.memory_allocated()/1e9:.2f} GB')
    break

# Verify teacher untouched
unchanged = all(torch.equal(s, p) for s, p in zip(snapshot, teacher.encoder.parameters()))
print(f'[smoke] frozen-teacher invariant: {"PASS" if unchanged else "FAIL — teacher params changed!"}')

## 7. Train Method 1 end-to-end

Saves to Drive (`MyDrive/v9c_crossjepa_method1/last.pt`) every 200 steps + end of epoch, so a Colab disconnect doesn't lose progress. Resumes automatically via `--resume auto`.

In [ ]:
!python src/train_v9c_method1_vol2slice.py --scans_glob '/content/data/healthy/*/sub-*/ses-*/anat/*.nii.gz' --v8_ckpt {V8_CKPT} --output_dir /content/drive/MyDrive/v9c_crossjepa_method1 --volume_size 144 192 192 --in_channels 1 --patch_size 16 --encoder_dim 384 --encoder_depth 12 --encoder_heads 6 --predictor_dim 192 --predictor_depth 6 --batch_size 2 --slices_per_volume 6 --epochs 30 --lr 2e-4 --num_workers 2 --amp --checkpoint_every_steps 200 --resume auto

## (Optional) Anomaly-eval Method 1 on BraTS

Once Method 1 has trained for ~20+ epochs, the model should produce HIGHER prediction-error on tumor volumes than on held-out healthy. This is the anomaly signal.

```python
from src.research.v9c_crossjepa.volume_to_slice import Vol2SliceModel
# load trained encoder + predictor, then for each BraTS volume compute
# model.anomaly_score(volume, slice, conditioning) and compare to the
# distribution on a held-out healthy validation set.
# (Wire-up shipped in a follow-up PR alongside the conformal calibration.)
```